# Gemini Labeling for Meeting Extraction

## Enhanced Temporal Extraction Pipeline

**What this notebook does:**
1. Loads your 4,172 meeting emails from `meeting_intent.jsonl`
2. Labels them with Gemini API (extracts title, attendees, start_utc, end_utc, location)
3. Uses enhanced prompting for better time extraction
4. Saves labeled data ready for fine-tuning

**Requirements:**
- Google Colab (A100 optional - not needed for labeling)
- Gemini API key
- meeting_intent.jsonl file uploaded

**Time:** 30-45 minutes  
**Cost:** ~$10-20

**Expected Output:**
- 4,172 fully labeled emails
- 30-40% with temporal information
- 100% with title and attendees

---

## STEP 1: Install Dependencies

In [1]:
# Install required packages
!pip install -q google-generativeai tqdm python-dateutil pytz

print(" Installation complete!")

 Installation complete!


## STEP 2: Import Libraries

In [2]:
import os
import json
import time
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from datetime import datetime, timedelta

import google.generativeai as genai
from tqdm import tqdm
from dateutil import parser as date_parser
import pytz

print(" All libraries imported successfully")

 All libraries imported successfully


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## STEP 3: Configure Gemini API

**Get your API key:** https://makersuite.google.com/app/apikey

Choose one of these methods:

In [ ]:
# ============================================
# METHOD 1: Direct API Key (Quick & Simple)
# ============================================
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")  # Replace with your actual key

# ============================================
# METHOD 2: Google Colab Secrets (Recommended)
# ============================================
# Uncomment these lines to use Colab Secrets:
# from google.colab import userdata
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# ============================================
# METHOD 3: Environment Variable
# ============================================
# GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

# Validate API key
if not GEMINI_API_KEY or GEMINI_API_KEY == "YOUR_GEMINI_API_KEY_HERE":
    print(" ERROR: Please set your GEMINI_API_KEY above")
    print("Get your key from: https://makersuite.google.com/app/apikey")
else:
    # Configure Gemini
    genai.configure(api_key=GEMINI_API_KEY)
    print(" Gemini API configured successfully")
    print(f" API Key: {GEMINI_API_KEY[:10]}...{GEMINI_API_KEY[-4:]}")

 Gemini API configured successfully
 API Key: AIzaSyAy0U...UAsY


## STEP 4: Upload meeting_intent.jsonl

**Option A:** Upload via Colab file browser (left sidebar)

**Option B:** Mount Google Drive (if file is there)

**Option C:** Run the cell below to upload

In [5]:
from google.colab import files

print("Please upload meeting_intent.jsonl file:")
uploaded = files.upload()

# Verify upload
if 'meeting_intent.jsonl' in uploaded:
    print(f" File uploaded: {len(uploaded['meeting_intent.jsonl'])} bytes")
    INPUT_FILE = "meeting_intent.jsonl"
else:
    print("meeting_intent.jsonl not found")
    print("Available files:", list(uploaded.keys()))

Please upload meeting_intent.jsonl file:


Saving meeting_intent.jsonl to meeting_intent.jsonl
 File uploaded: 3085700 bytes


## STEP 5: Configuration

In [6]:
# ============================================
# Configuration Settings
# ============================================

class Config:
    """Labeling configuration."""

    # Input/Output
    INPUT_FILE = "meeting_intent.jsonl"  # Your uploaded file
    OUTPUT_DIR = "./output"
    OUTPUT_FILE = "labeled_meeting_emails.jsonl"
    CHECKPOINT_FILE = "labeling_checkpoint.jsonl"

    # Gemini settings
    MODEL_NAME = "gemini-2.5-flash"  # Fast and cheap
    TEMPERATURE = 0.0  # Deterministic output

    # Batch processing
    BATCH_SIZE = 10  # Process 10 emails at a time
    USE_BATCH = True  # Use batch API (faster)

    # Rate limiting
    BASE_SLEEP = 1.0  # Seconds between batches
    JITTER = 0.5  # Random jitter to avoid rate limits
    MAX_RETRIES = 3  # Retry failed requests

    # Resume capability
    USE_CHECKPOINTING = True  # Save progress for resume

# Create output directory
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"Configuration set")
print(f"  Input: {Config.INPUT_FILE}")
print(f"  Output: {Config.OUTPUT_DIR}/{Config.OUTPUT_FILE}")
print(f"  Model: {Config.MODEL_NAME}")
print(f"  Batch size: {Config.BATCH_SIZE}")

Configuration set
  Input: meeting_intent.jsonl
  Output: ./output/labeled_meeting_emails.jsonl
  Model: gemini-2.5-flash
  Batch size: 10


## STEP 6: Load Input Data

In [7]:
def load_meeting_emails(filepath: str) -> List[Dict]:
    """
    Load meeting emails from JSONL file.

    Args:
        filepath: Path to meeting_intent.jsonl

    Returns:
        List of email dictionaries
    """
    emails = []

    print(f"Loading emails from {filepath}...")

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                email = json.loads(line)
                emails.append(email)
            except json.JSONDecodeError as e:
                print(f"Warning: Failed to parse line: {e}")
                continue

    print(f"✓ Loaded {len(emails)} emails")
    return emails


# Load the data
meeting_emails = load_meeting_emails(Config.INPUT_FILE)

# Show sample
if meeting_emails:
    print("\nSample email:")
    sample = meeting_emails[0]
    print(f"  Email ID: {sample.get('email_id', 'N/A')[:50]}...")
    print(f"  Subject: {sample.get('subject', '(empty)')[:80]}")
    print(f"  Body length: {len(sample.get('body', ''))} chars")
    print(f"  Label: {sample.get('label', 'N/A')}")

Loading emails from meeting_intent.jsonl...
✓ Loaded 4172 emails

Sample email:
  Email ID: <29177675.1075855687692.JavaMail.evans@thyme>...
  Subject: meeting re: storage strategies in the west
  Body length: 647 chars
  Label: 1


## STEP 7: Checkpoint Management

This allows resuming if the process is interrupted.

In [8]:
def load_checkpoint(checkpoint_file: str) -> set:
    """
    Load already-processed email IDs from checkpoint.

    Args:
        checkpoint_file: Path to checkpoint JSONL

    Returns:
        Set of processed email IDs
    """
    done_ids = set()

    checkpoint_path = os.path.join(Config.OUTPUT_DIR, checkpoint_file)

    if not os.path.exists(checkpoint_path):
        return done_ids

    with open(checkpoint_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                email_id = obj.get('email_id')
                if email_id:
                    done_ids.add(str(email_id))
            except:
                continue

    return done_ids


def save_to_checkpoint(email_id: str, labels: Dict, checkpoint_file: str):
    """
    Append labeled email to checkpoint file.

    Args:
        email_id: Email identifier
        labels: Extracted labels
        checkpoint_file: Path to checkpoint file
    """
    checkpoint_path = os.path.join(Config.OUTPUT_DIR, checkpoint_file)

    with open(checkpoint_path, 'a', encoding='utf-8') as f:
        obj = {'email_id': email_id, 'labels': labels}
        f.write(json.dumps(obj, ensure_ascii=False) + '\n')


# Load existing checkpoint
done_ids = load_checkpoint(Config.CHECKPOINT_FILE)
print(f"✓ Checkpoint loaded: {len(done_ids)} emails already processed")

✓ Checkpoint loaded: 3380 emails already processed


## STEP 8: Enhanced Labeling Prompt

This prompt includes improved temporal extraction logic.

In [9]:
def create_enhanced_labeling_prompt(email: Dict) -> str:
    """
    Create enhanced prompt with temporal reasoning.

    Args:
        email: Email dictionary with subject and body

    Returns:
        Formatted prompt string
    """
    subject = email.get('subject', '(empty)')
    body = email.get('body', '')
    email_text = f"Subject: {subject}\n\n{body}"

    prompt = f"""You are a meeting information extraction system specialized in temporal reasoning.

Extract meeting details from this email:

---
{email_text}
---

Return ONLY a JSON object with these exact fields:
{{
  "title": "meeting subject or description",
  "attendees": ["email1@domain.com", "email2@domain.com"],
  "start_utc": "ISO 8601 UTC format or null",
  "end_utc": "ISO 8601 UTC format or null",
  "location": "meeting location or null",
  "time_confidence": "high|medium|low|none"
}}

EXTRACTION RULES:

1. TITLE (required):
   - Use subject line if available
   - Otherwise use meeting description from body
   - If labeled "Description:" use that
   - Make it concise (max 100 chars)

2. ATTENDEES (required):
   - Extract all email addresses mentioned
   - Look in To:, Cc:, attendee lists
   - Format: lowercase email addresses
   - Return empty list if none found: []

3. TEMPORAL EXTRACTION:

   HIGH CONFIDENCE (structured format):
   - "Start: 10/03/2000 02:30 PM" → Parse and convert to UTC
   - "End: 10/03/2000 03:30 PM" → Parse and convert to UTC
   - Assume CST timezone (UTC-6) if not specified
   - time_confidence: "high"

   MEDIUM CONFIDENCE (explicit date + time):
   - "Tuesday, Oct. 10th at 4:00pm" → Parse date and time
   - "10/10/2000 at 4:00pm" → Convert to UTC
   - Assume CST timezone
   - time_confidence: "medium"

   LOW CONFIDENCE (incomplete):
   - Only date mentioned: "October 10th" → start_utc: null
   - Only time mentioned: "2pm" → start_utc: null
   - Vague: "Thursday afternoon" → start_utc: null
   - time_confidence: "low"

   NO TEMPORAL INFO:
   - "Let's meet soon" → start_utc: null, end_utc: null
   - "We should schedule something" → start_utc: null, end_utc: null
   - time_confidence: "none"

4. END TIME:
   - If explicit End time given, extract it
   - If duration mentioned: "1 hour meeting" → calculate from start
   - Otherwise: end_utc: null

5. LOCATION:
   - Room numbers: "Room 2537", "EB3270"
   - Conference rooms: "Conference Room B"
   - Virtual: "Zoom", "Teams" (extract link if present)
   - If no location: location: null

6. TIME CONFIDENCE:
   - "high": Exact date + time with clear format
   - "medium": Date + time but requires parsing
   - "low": Only partial temporal info
   - "none": No extractable time

TIMEZONE CONVERSION:
- Enron emails are typically CST (Central Standard Time = UTC-6)
- Convert all times to UTC
- Format: "YYYY-MM-DDTHH:MM:SSZ"
- Example: "10/03/2000 02:30 PM CST" → "2000-10-03T20:30:00Z"

IMPORTANT:
- Return ONLY the JSON object
- No markdown formatting (no ```json)
- No explanatory text before or after
- All fields must be present (use null if not found)
- Attendees must be a list (even if empty: [])

JSON:"""

    return prompt


# Test the prompt
if meeting_emails:
    test_prompt = create_enhanced_labeling_prompt(meeting_emails[0])
    print(" Prompt created successfully")
    print(f"  Prompt length: {len(test_prompt)} chars")

 Prompt created successfully
  Prompt length: 3396 chars


## STEP 9: Labeling Functions

In [10]:
def extract_json_from_response(response_text: str) -> str:
    """
    Clean Gemini response to extract JSON.

    Args:
        response_text: Raw Gemini response

    Returns:
        Cleaned JSON string
    """
    text = response_text.strip()

    # Remove markdown code blocks if present
    if text.startswith('```'):
        text = text.replace('```json', '').replace('```', '')

    # Find JSON object
    start = text.find('{')
    end = text.rfind('}') + 1

    if start >= 0 and end > start:
        return text[start:end]

    return text


def validate_labels(labels: Dict) -> Tuple[bool, str]:
    """
    Validate extracted labels.

    Args:
        labels: Extracted labels dictionary

    Returns:
        Tuple of (is_valid, error_message)
    """
    required_fields = ['title', 'attendees', 'start_utc', 'end_utc', 'location', 'time_confidence']

    # Check all fields present
    for field in required_fields:
        if field not in labels:
            return False, f"Missing field: {field}"

    # Validate attendees is a list
    if not isinstance(labels['attendees'], list):
        return False, "attendees must be a list"

    # Validate time_confidence
    valid_confidence = ['high', 'medium', 'low', 'none']
    if labels['time_confidence'] not in valid_confidence:
        labels['time_confidence'] = 'none'  # Fix it

    return True, "valid"


def label_single_email(email: Dict, model) -> Tuple[Optional[Dict], Optional[str]]:
    """
    Label a single email with Gemini.

    Args:
        email: Email dictionary
        model: Gemini model instance

    Returns:
        Tuple of (labels_dict, error_message)
    """
    try:
        # Create prompt
        prompt = create_enhanced_labeling_prompt(email)

        # Call Gemini
        response = model.generate_content(
            prompt,
            generation_config={'temperature': Config.TEMPERATURE}
        )

        # Extract JSON
        response_text = extract_json_from_response(response.text)
        labels = json.loads(response_text)

        # Validate
        is_valid, error = validate_labels(labels)
        if not is_valid:
            return None, error

        return labels, None

    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {str(e)}"
    except Exception as e:
        return None, f"Labeling error: {str(e)}"


print("✓ Labeling functions defined")

✓ Labeling functions defined


## STEP 10: Batch Processing (Optional but Faster)

In [11]:
def label_batch(emails: List[Dict], model) -> List[Tuple[Optional[Dict], Optional[str]]]:
    """
    Label a batch of emails (sequential fallback if batch API unavailable).

    Args:
        emails: List of email dictionaries
        model: Gemini model instance

    Returns:
        List of (labels, error) tuples
    """
    results = []

    # Process sequentially (batch API can be unreliable)
    for email in emails:
        labels, error = label_single_email(email, model)
        results.append((labels, error))

    return results


print(" Batch processing defined")

 Batch processing defined


STEP 11 : PIPELINE

In [12]:
# ============================================
# COMPLETE: DRIVE BACKUP + CHECKPOINT RELOAD + LABELING PIPELINE
# Run this ONE cell after cells 1-8 (setup, imports, functions)
# ============================================

from google.colab import drive
import shutil
import os

# ============================================
# PART 1: GOOGLE DRIVE BACKUP SETUP
# ============================================

print("=" * 80)
print("SETTING UP GOOGLE DRIVE BACKUP (Every 10 emails)")
print("=" * 80)

# Mount Google Drive
print("\nMounting Google Drive...")
drive.mount('/content/drive', force_remount=False)
print("✓ Google Drive mounted")

# Create backup directory
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/meeting_extraction_backup'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"✓ Backup folder: {DRIVE_BACKUP_DIR}")

# Check for existing backup and restore
DRIVE_CHECKPOINT = os.path.join(DRIVE_BACKUP_DIR, 'labeling_checkpoint.jsonl')
LOCAL_CHECKPOINT = os.path.join(Config.OUTPUT_DIR, Config.CHECKPOINT_FILE)

if os.path.exists(DRIVE_CHECKPOINT):
    print(f"\n✓ Found backup in Drive - restoring...")
    os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
    shutil.copy2(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)
    with open(LOCAL_CHECKPOINT, 'r') as f:
        count = sum(1 for _ in f)
    print(f"✓ Restored {count:,} emails from backup")
else:
    print(f"\n⚠️  No existing backup found - starting fresh")

# Modify save function to backup every 10 emails
original_save = save_to_checkpoint

def save_with_backup(email_id: str, labels: Dict, checkpoint_file: str):
    """Save locally + backup to Drive every 10 emails."""

    # Always save locally first
    original_save(email_id, labels, checkpoint_file)

    # Backup to Drive every 10 emails
    if not hasattr(save_with_backup, 'counter'):
        save_with_backup.counter = 0

    save_with_backup.counter += 1

    if save_with_backup.counter % 10 == 0:
        local_path = os.path.join(Config.OUTPUT_DIR, checkpoint_file)
        drive_path = os.path.join(DRIVE_BACKUP_DIR, checkpoint_file)
        shutil.copy2(local_path, drive_path)

# Replace the function
save_to_checkpoint = save_with_backup

print("\n✓ Backup configured!")
print(f"  Local saves: Every email")
print(f"  Drive backup: Every 10 emails")
print(f"  Location: {DRIVE_BACKUP_DIR}")


# ============================================
# PART 2: RELOAD CHECKPOINT (CRITICAL!)
# ============================================

print("\n" + "=" * 80)
print("RELOADING CHECKPOINT")
print("=" * 80)

# Reload the checkpoint (now includes Drive-restored emails)
done_ids = load_checkpoint(Config.CHECKPOINT_FILE)

print(f"\n✓ Checkpoint reloaded")
print(f"  Already processed: {len(done_ids):,} emails")

if len(done_ids) > 0:
    print(f"  Status: RESUMING from previous run")
else:
    print(f"  Status: STARTING fresh")


# ============================================
# PART 3: MAIN LABELING PIPELINE
# ============================================

def run_labeling_pipeline(
    emails: List[Dict],
    done_ids: set,
    checkpoint_file: str,
    output_file: str
) -> List[Dict]:
    """
    Main labeling pipeline with checkpointing and error handling.
    Now includes automatic Google Drive backup every 10 emails.

    Args:
        emails: List of emails to label
        done_ids: Set of already-processed email IDs
        checkpoint_file: Checkpoint file name
        output_file: Final output file name

    Returns:
        List of successfully labeled emails
    """
    print("\n" + "="*80)
    print("STARTING LABELING PIPELINE")
    print("="*80)

    # Initialize Gemini model
    print(f"\nInitializing {Config.MODEL_NAME}...")
    model = genai.GenerativeModel(Config.MODEL_NAME)
    print("✓ Model initialized")

    # Filter out already-done emails
    queue = []
    for email in emails:
        email_id = str(email.get('email_id', ''))
        if email_id not in done_ids:
            queue.append(email)

    print(f"\nProcessing status:")
    print(f"  Total emails: {len(emails)}")
    print(f"  Already done: {len(done_ids)}")
    print(f"  Remaining: {len(queue)}")
    print(f"  Batch size: {Config.BATCH_SIZE}")
    print(f"  Estimated batches: {(len(queue) + Config.BATCH_SIZE - 1) // Config.BATCH_SIZE}")
    print(f"\n✓ Google Drive backup: Every 10 emails")
    print(f"  Backup location: {DRIVE_BACKUP_DIR}")

    if not queue:
        print("\n✓ All emails already labeled!")
        return []

    # Statistics
    stats = {
        'total': len(queue),
        'success': 0,
        'failed': 0,
        'with_times': 0
    }

    labeled_emails = []

    # Process in batches
    print("\nStarting labeling...")
    if len(queue) > 1000:
        print(f"This will take approximately {len(queue)*6.85/60:.0f} minutes for {len(queue):,} emails")
    else:
        print(f"This will take approximately {len(queue)*6.85/60:.0f} minutes for {len(queue):,} emails")
    print("Progress will be saved continuously - safe to interrupt\n")

    pbar = tqdm(total=len(queue), desc="Labeling emails")

    for i in range(0, len(queue), Config.BATCH_SIZE):
        batch = queue[i:i + Config.BATCH_SIZE]

        # Label batch
        results = label_batch(batch, model)

        # Process results
        for email, (labels, error) in zip(batch, results):
            email_id = str(email.get('email_id', f"email_{i}"))

            if labels:
                # Success
                labeled_email = {
                    'email_id': email_id,
                    'subject': email.get('subject', ''),
                    'body': email.get('body', ''),
                    'original_label': email.get('label', 1),
                    'extracted': labels
                }

                labeled_emails.append(labeled_email)
                stats['success'] += 1

                # Track temporal extraction
                if labels.get('start_utc'):
                    stats['with_times'] += 1

                # Save to checkpoint (now includes Drive backup every 10)
                if Config.USE_CHECKPOINTING:
                    save_to_checkpoint(email_id, labels, checkpoint_file)
            else:
                # Failed
                stats['failed'] += 1

            pbar.update(1)

        # Rate limiting
        if i + Config.BATCH_SIZE < len(queue):
            sleep_time = Config.BASE_SLEEP + random.uniform(0, Config.JITTER)
            time.sleep(sleep_time)

    pbar.close()

    # Print statistics
    print("\n" + "="*80)
    print("LABELING COMPLETE")
    print("="*80)
    print(f"\nResults:")
    print(f"  Successfully labeled: {stats['success']} ({stats['success']/stats['total']*100:.1f}%)")
    print(f"  Failed: {stats['failed']} ({stats['failed']/stats['total']*100:.1f}%)")
    if stats['success'] > 0:
        print(f"  With temporal info: {stats['with_times']} ({stats['with_times']/stats['success']*100:.1f}% of success)")

    # Save final output
    output_path = os.path.join(Config.OUTPUT_DIR, output_file)
    with open(output_path, 'w', encoding='utf-8') as f:
        for email in labeled_emails:
            f.write(json.dumps(email, ensure_ascii=False) + '\n')

    print(f"\n✓ Saved {len(labeled_emails)} labeled emails to: {output_path}")

    # Also save final output to Drive
    drive_output_path = os.path.join(DRIVE_BACKUP_DIR, output_file)
    shutil.copy2(output_path, drive_output_path)
    print(f"✓ Backup saved to Google Drive: {drive_output_path}")

    return labeled_emails


# ============================================
# RUN THE PIPELINE
# ============================================

labeled_data = run_labeling_pipeline(
    emails=meeting_emails,
    done_ids=done_ids,
    checkpoint_file=Config.CHECKPOINT_FILE,
    output_file=Config.OUTPUT_FILE
)

print("\n" + "=" * 80)
print("🎉 ALL DONE! 🎉")
print("=" * 80)


SETTING UP GOOGLE DRIVE BACKUP (Every 10 emails)

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted
✓ Backup folder: /content/drive/MyDrive/meeting_extraction_backup

✓ Found backup in Drive - restoring...
✓ Restored 3,380 emails from backup

✓ Backup configured!
  Local saves: Every email
  Drive backup: Every 10 emails
  Location: /content/drive/MyDrive/meeting_extraction_backup

RELOADING CHECKPOINT

✓ Checkpoint reloaded
  Already processed: 3,380 emails
  Status: RESUMING from previous run

STARTING LABELING PIPELINE

Initializing gemini-2.5-flash...
✓ Model initialized

Processing status:
  Total emails: 4172
  Already done: 3380
  Remaining: 792
  Batch size: 10
  Estimated batches: 80

✓ Google Drive backup: Every 10 emails
  Backup location: /content/drive/MyDrive/meeting_extraction_backup

Starting labeling...
This will take approximately 90 minutes f

Labeling emails: 100%|██████████| 792/792 [1:28:58<00:00,  6.74s/it]


LABELING COMPLETE

Results:
  Successfully labeled: 792 (100.0%)
  Failed: 0 (0.0%)
  With temporal info: 352 (44.4% of success)

✓ Saved 792 labeled emails to: ./output/labeled_meeting_emails.jsonl
✓ Backup saved to Google Drive: /content/drive/MyDrive/meeting_extraction_backup/labeled_meeting_emails.jsonl

🎉 ALL DONE! 🎉


## STEP 12: Analyze Results

In [13]:
def analyze_labeled_data(labeled_emails: List[Dict]):
    """
    Analyze quality and completeness of labeled data.

    Args:
        labeled_emails: List of labeled email dictionaries
    """
    if not labeled_emails:
        print("No labeled emails to analyze")
        return

    print("\n" + "="*80)
    print("LABEL QUALITY ANALYSIS")
    print("="*80)

    total = len(labeled_emails)

    # Field completeness
    has_title = sum(1 for e in labeled_emails if e['extracted'].get('title'))
    has_attendees = sum(1 for e in labeled_emails if e['extracted'].get('attendees'))
    has_start = sum(1 for e in labeled_emails if e['extracted'].get('start_utc'))
    has_end = sum(1 for e in labeled_emails if e['extracted'].get('end_utc'))
    has_location = sum(1 for e in labeled_emails if e['extracted'].get('location'))

    print(f"\nField Completeness (n={total}):")
    print(f"  Title: {has_title} ({has_title/total*100:.1f}%)")
    print(f"  Attendees: {has_attendees} ({has_attendees/total*100:.1f}%)")
    print(f"  Start time: {has_start} ({has_start/total*100:.1f}%)")
    print(f"  End time: {has_end} ({has_end/total*100:.1f}%)")
    print(f"  Location: {has_location} ({has_location/total*100:.1f}%)")

    # Attendee statistics
    attendee_counts = [len(e['extracted'].get('attendees', [])) for e in labeled_emails]
    avg_attendees = sum(attendee_counts) / len(attendee_counts) if attendee_counts else 0

    print(f"\nAttendee Statistics:")
    print(f"  Average per email: {avg_attendees:.1f}")
    print(f"  Min: {min(attendee_counts) if attendee_counts else 0}")
    print(f"  Max: {max(attendee_counts) if attendee_counts else 0}")

    # Time confidence distribution
    confidence_dist = {}
    for email in labeled_emails:
        conf = email['extracted'].get('time_confidence', 'none')
        confidence_dist[conf] = confidence_dist.get(conf, 0) + 1

    print(f"\nTime Confidence Distribution:")
    for conf, count in sorted(confidence_dist.items()):
        print(f"  {conf}: {count} ({count/total*100:.1f}%)")

    # Show examples
    print("\n" + "="*80)
    print("SAMPLE LABELED EMAILS")
    print("="*80)

    # Example with full info
    full_example = None
    for email in labeled_emails:
        if (email['extracted'].get('start_utc') and
            email['extracted'].get('location') and
            len(email['extracted'].get('attendees', [])) > 0):
            full_example = email
            break

    if full_example:
        print("\nExample: Complete extraction")
        print(f"Subject: {full_example['subject'][:80]}")
        print(f"Extracted:")
        print(json.dumps(full_example['extracted'], indent=2))

    # Example without time
    no_time_example = None
    for email in labeled_emails:
        if not email['extracted'].get('start_utc'):
            no_time_example = email
            break

    if no_time_example:
        print("\nExample: No temporal information")
        print(f"Subject: {no_time_example['subject'][:80]}")
        print(f"Extracted:")
        print(json.dumps(no_time_example['extracted'], indent=2))

    print("\n" + "="*80)


# Analyze the results
if labeled_data:
    analyze_labeled_data(labeled_data)
else:
    print("No new labels to analyze (all emails already processed)")


LABEL QUALITY ANALYSIS

Field Completeness (n=792):
  Title: 792 (100.0%)
  Attendees: 365 (46.1%)
  Start time: 352 (44.4%)
  End time: 168 (21.2%)
  Location: 414 (52.3%)

Attendee Statistics:
  Average per email: 1.7
  Min: 0
  Max: 72

Time Confidence Distribution:
  high: 66 (8.3%)
  low: 317 (40.0%)
  medium: 287 (36.2%)
  none: 122 (15.4%)

SAMPLE LABELED EMAILS

Example: Complete extraction
Subject: puget sound requests nw price caps
Extracted:
{
  "title": "puget sound requests nw price caps",
  "attendees": [
    "elizabeth.sager@ect.com",
    "mark.taylor@ect.com",
    "richard.b.sanders@ect.com",
    "mark.e.haedicke@ect.com"
  ],
  "start_utc": "2000-10-30T17:30:00Z",
  "end_utc": null,
  "location": "Conference Call",
  "time_confidence": "medium"
}

Example: No temporal information
Subject: ena customer meeting
Extracted:
{
  "title": "ENA Customer Meeting",
  "attendees": [
    "lynn.blair@enron.com"
  ],
  "start_utc": null,
  "end_utc": null,
  "location": "Miami",
 

## STEP 13: Download Results

Download the labeled data to your computer.

In [14]:
from google.colab import files

# Download the labeled file
output_path = os.path.join(Config.OUTPUT_DIR, Config.OUTPUT_FILE)

if os.path.exists(output_path):
    print(f"Downloading: {output_path}")
    files.download(output_path)
    print(" Download started")
else:
    print(f" File not found: {output_path}")

Downloading: ./output/labeled_meeting_emails.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Download started
